In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

In [1]:
pwd

'/Users/abannee/Documents/GitHub/fraud_detection_ml/notebooks'

In [3]:
train_txn = pd.read_csv('/Users/abannee/Documents/GitHub/fraud_detection_ml/data/raw/ieee-fraud-detection/train_transaction.csv')
train_id  = pd.read_csv('/Users/abannee/Documents/GitHub/fraud_detection_ml/data/raw/ieee-fraud-detection/train_identity.csv')

print("Transaction shape:", train_txn.shape)
print("Identity shape   :", train_id.shape)

Transaction shape: (590540, 394)
Identity shape   : (144233, 41)


In [4]:
# How many transactions have identity records?
joined = train_txn['TransactionID'].isin(train_id['TransactionID'])
print(f"Transactions WITH identity : {joined.sum():,}  ({joined.mean()*100:.1f}%)")
print(f"Transactions WITHOUT identity: {(~joined).sum():,}  ({(~joined).mean()*100:.1f}%)")

Transactions WITH identity : 144,233  (24.4%)
Transactions WITHOUT identity: 446,307  (75.6%)


In [5]:
df = train_txn.merge(train_id, on='TransactionID', how='left')
print("Merged shape:", df.shape)

Merged shape: (590540, 434)


In [6]:
fraud_rate = df['isFraud'].value_counts(normalize=True)
print(fraud_rate)
print(f"\nFraud count : {df['isFraud'].sum():,}")
print(f"Total rows  : {len(df):,}")

isFraud
0   0.9650
1   0.0350
Name: proportion, dtype: float64

Fraud count : 20,663
Total rows  : 590,540


In [7]:
null_summary = pd.DataFrame({
    'null_count'  : df.isnull().sum(),
    'null_pct'    : (df.isnull().sum() / len(df) * 100).round(2),
    'dtype'       : df.dtypes,
    'nunique'     : df.nunique(),
    'sample_val'  : df.iloc[0]
}).sort_values('null_pct', ascending=False)

print(null_summary.head(50))

       null_count  null_pct    dtype  nunique sample_val
id_24      585793   99.2000  float64       12        NaN
id_25      585408   99.1300  float64      341        NaN
id_26      585377   99.1300  float64       95        NaN
id_21      585381   99.1300  float64      490        NaN
id_08      585385   99.1300  float64       94        NaN
id_07      585385   99.1300  float64       84        NaN
id_27      585371   99.1200      str        2        NaN
id_23      585371   99.1200      str        3        NaN
id_22      585371   99.1200  float64       25        NaN
dist2      552913   93.6300  float64     1751        NaN
D7         551623   93.4100  float64      597        NaN
id_18      545427   92.3600  float64       18        NaN
D13        528588   89.5100  float64      577        NaN
D14        528353   89.4700  float64      802        NaN
D12        525823   89.0400  float64      635        NaN
id_03      524216   88.7700  float64       24        NaN
id_04      524216   88.7700  fl

In [8]:
feature_groups = {
    'V_grp9'  : [f'V{i}' for i in range(217, 279)],
    'V_grp11' : [f'V{i}' for i in range(322, 340)],
    'V_grp6'  : [f'V{i}' for i in range(95, 138)],
    'V_grp10' : [f'V{i}' for i in range(279, 322)],
    'D_cols'  : [f'D{i}' for i in range(1, 16)],
    'C_cols'  : [f'C{i}' for i in range(1, 15)],
    'M_cols'  : [f'M{i}' for i in range(1, 10)],
    'id_num'  : [f'id_0{i}' if i < 10 else f'id_{i}' for i in range(1, 12)],
    'id_cat'  : [f'id_{i}' for i in range(12, 39)],
}

for grp, cols in feature_groups.items():
    existing = [c for c in cols if c in df.columns]
    avg_null = df[existing].isnull().mean().mean() * 100
    print(f"{grp:12s} | cols: {len(existing):3d} | avg null%: {avg_null:.1f}%")

V_grp9       | cols:  62 | avg null%: 77.4%
V_grp11      | cols:  18 | avg null%: 86.1%
V_grp6       | cols:  43 | avg null%: 0.1%
V_grp10      | cols:  43 | avg null%: 0.1%
D_cols       | cols:  15 | avg null%: 58.2%
C_cols       | cols:  14 | avg null%: 0.0%
M_cols       | cols:   9 | avg null%: 49.9%
id_num       | cols:  11 | avg null%: 84.7%
id_cat       | cols:  27 | avg null%: 84.9%


In [9]:
print("\n--- Numeric columns ---")
print(df.select_dtypes(include='number').columns.tolist())

print("\n--- Categorical/Object columns ---")
print(df.select_dtypes(include='object').columns.tolist())


--- Numeric columns ---
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V88', 'V89', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95'

/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_31278/1980594097.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include='object').columns.tolist())


In [10]:
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage: {mem_mb:.1f} MB")

# Downcast to reduce memory
for col in df.select_dtypes(include='float64').columns:
    df[col] = pd.to_numeric(df[col], downcast='float')
for col in df.select_dtypes(include='int64').columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')

print(f"After downcast: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")

Memory usage: 2514.0 MB
After downcast: 1603.3 MB


In [14]:
df.to_pickle('/Users/abannee/Documents/GitHub/fraud_detection_ml/models/ieee_base.pkl')
print("Saved ieee_base.pkl")

Saved ieee_base.pkl
